# Camada Bronze - Women's Clothing Reviews + Rent the Runway + US Holiday Dates

Carrega os ficheiros originais para o HDFS e cria as tabelas Bronze em formato Delta.
Os dados ficam tal como vêm da fonte (todas as colunas como texto, com os nomes originais),
acrescidos de metadados de ingestão: `_data_ingestao` e `_ficheiro_origem`.

| Dataset | Ficheiro | Tabela Bronze |
|---|---|---|
| Women's Clothing (principal) | Womens Clothing E-Commerce Reviews.csv | bronze.bronze_reviews |
| Rent the Runway (complementar) | renttherunway_final_data.json | bronze.bronze_rtr_reviews |
| US Holiday Dates (complementar dos colegas) | US Holiday Dates (2004-2021).csv | bronze.bronze_us_holidays |

A Bronze guarda todas as datas; o filtro do intervalo de tempo do grupo (junho a dezembro de 2017) é feito na Silver.

In [1]:
# Copy files to bronze layer (HDFS)
from hdfs import InsecureClient
client = InsecureClient("http://hdfs-nn:9870", user="anonymous")

ficheiros = {
    "./Womens Clothing E-Commerce Reviews.csv": "/projeto/bronze/womens_clothing/womens_clothing_reviews.csv",
    "./data/rtr/renttherunway_final_data.json": "/projeto/bronze/rtr/renttherunway_final_data.json",
    "./data/holidays/US Holiday Dates (2004-2021).csv": "/projeto/bronze/holidays/us_holiday_dates_2004_2021.csv",
}

# Full load: apaga a versão anterior e volta a carregar
for from_path, to_path in ficheiros.items():
    client.delete(to_path)
    client.upload(to_path, from_path)
    print(to_path, client.status(to_path)["length"], "bytes")

/projeto/bronze/womens_clothing/womens_clothing_reviews.csv 8483448 bytes
/projeto/bronze/rtr/renttherunway_final_data.json 123251881 bytes
/projeto/bronze/holidays/us_holiday_dates_2004_2021.csv 15699 bytes


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta import *

# warehouse_location points to the default location for managed databases and tables
warehouse_location = 'hdfs://hdfs-nn:9000/warehouse'

builder = SparkSession \
    .builder \
    .appName("Bronze - Womens Clothing + Rent the Runway + US Holidays") \
    .config("spark.sql.warehouse.dir", warehouse_location) \
    .config("hive.metastore.uris", "thrift://hive-metastore:9083") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.jars.packages", "io.delta:delta-core_2.12:2.4.0") \
    .enableHiveSupport()

spark = configure_spark_with_delta_pip(builder).getOrCreate()

# Os nomes originais das colunas têm espaços ("Clothing ID", "bust size", "rented for");
# o column mapping do Delta permite mantê-los
spark.conf.set("spark.databricks.delta.properties.defaults.columnMapping.mode", "name")
spark.conf.set("spark.databricks.delta.properties.defaults.minReaderVersion", "2")
spark.conf.set("spark.databricks.delta.properties.defaults.minWriterVersion", "5")

In [3]:
spark.sql("CREATE DATABASE IF NOT EXISTS bronze LOCATION 'hdfs://hdfs-nn:9000/warehouse/bronze.db/'")

DataFrame[]

In [4]:
def gravar_bronze(df, tabela):
    """Acrescenta os metadados de ingestão e grava a tabela Delta tal como veio da fonte."""
    df = df.withColumn("_data_ingestao", F.current_timestamp()) \
           .withColumn("_ficheiro_origem", F.input_file_name())
    df.write \
        .mode("overwrite") \
        .format("delta") \
        .option("overwriteSchema", "true") \
        .option("path", f"hdfs://hdfs-nn:9000/warehouse/bronze.db/{tabela}/") \
        .saveAsTable(f"bronze.{tabela}")
    print(f"bronze.{tabela}: {spark.table(f'bronze.{tabela}').count()} linhas, {len(df.columns) - 2} colunas originais")

def ler_csv(hdfs_path):
    return spark.read \
        .option("header", "true") \
        .option("multiLine", "true") \
        .option("escape", '"') \
        .csv("hdfs://hdfs-nn:9000" + hdfs_path)

# Women's Clothing: a 1.ª coluna não tem nome no CSV (o Spark chama-lhe _c0)
gravar_bronze(ler_csv("/projeto/bronze/womens_clothing/womens_clothing_reviews.csv").withColumnRenamed("_c0", "Unnamed: 0"),
              "bronze_reviews")

# Rent the Runway: JSON com uma avaliação por linha; todos os campos ficam como texto
rtr = spark.read.option("primitivesAsString", "true").json("hdfs://hdfs-nn:9000/projeto/bronze/rtr/renttherunway_final_data.json")
gravar_bronze(rtr, "bronze_rtr_reviews")

# US Holiday Dates
gravar_bronze(ler_csv("/projeto/bronze/holidays/us_holiday_dates_2004_2021.csv"), "bronze_us_holidays")

bronze.bronze_reviews: 23486 linhas, 11 colunas originais
bronze.bronze_rtr_reviews: 192544 linhas, 15 colunas originais
bronze.bronze_us_holidays: 342 linhas, 6 colunas originais


## Verificação

Confirma que as tabelas Bronze têm o mesmo número de linhas dos ficheiros originais
(Women's Clothing 23 486; Rent the Runway 192 544; US Holiday Dates 342).

In [5]:
esperado = {"bronze_reviews": 23486, "bronze_rtr_reviews": 192544, "bronze_us_holidays": 342}
for tabela, n in esperado.items():
    real = spark.table(f"bronze.{tabela}").count()
    print(f"{tabela:22s} esperado {n:>7}  obtido {real:>7}  {'OK' if real == n else 'DIFERENTE'}")

bronze_reviews         esperado   23486  obtido   23486  OK
bronze_rtr_reviews     esperado  192544  obtido  192544  OK
bronze_us_holidays     esperado     342  obtido     342  OK


In [6]:
spark.sql("SHOW TABLES FROM bronze").show(truncate=False)
spark.table("bronze.bronze_rtr_reviews").select("item_id", "rating", "category", "fit", "review_date").show(3, truncate=False)
spark.table("bronze.bronze_us_holidays").show(3, truncate=False)

+---------+------------------+-----------+
|namespace|tableName         |isTemporary|
+---------+------------------+-----------+
|bronze   |bronze_reviews    |false      |
|bronze   |bronze_rtr_reviews|false      |
|bronze   |bronze_us_holidays|false      |
+---------+------------------+-----------+

+-------+------+--------+-----+----------------+
|item_id|rating|category|fit  |review_date     |
+-------+------+--------+-----+----------------+
|1226293|10    |shift   |fit  |June 17, 2014   |
|132738 |10    |gown    |large|December 9, 2013|
|2673874|10    |jumpsuit|fit  |May 28, 2015    |
+-------+------+--------+-----+----------------+
only showing top 3 rows

+----------+-----------+-------+-----+---+----+--------------------------+--------------------------------------------------------------------------+
|Date      |Holiday    |WeekDay|Month|Day|Year|_data_ingestao            |_ficheiro_origem                                                          |
+----------+-----------+------

In [7]:
spark.stop()